# Day 2 FOXF1 / BMP4 reporters — 01_dapi_whole_organoid_mask_review

**Feeds:** Fig 2g, through notebook `03`

**Position in the chain:** run `00`, `01`, `02`, `03` in order, from this lane's directory (`imaging/foxf1_bmp4_day2_fig2g/`).

Ported from the original analysis `FOXF1_BMP4_day2_expression/notebooks/01_dapi_whole_organoid_mask_review.ipynb`.

**Changes from the original notebook**
1. No code cell of the original was edited.
2. The notebook ships without outputs, as the original notebook file does. The original analysis also saved an executed copy with outputs, but that copy ran an earlier setting, `BACKGROUND_ESTIMATOR = "annulus"`. This version, with `"whole_off_organoid"`, is the one that wrote the manifest, masks and plane metrics that notebooks `02` and `03` read; its run was not saved with outputs.
3. CZI files are read through `src/trunk_morph_ref/czi_compat.py` instead of `czifile`, a change of one import in `scripts/day2_quantification_helpers.py`.


            # 01 | DAPI Whole-Organoid Mask Review

            ## Notebook Scope

            This notebook runs the current first-pass quantification pipeline,
            writes DAPI whole-organoid masks for every z plane,
            and gives image-heavy QC for mask adequacy across the dataset.
            

In [ ]:
import os
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
if (CWD / "scripts").exists() and (CWD / "results").exists():
    ROOT = CWD
elif (CWD.parent / "scripts").exists() and (CWD.parent / "results").exists():
    ROOT = CWD.parent.resolve()
else:
    ROOT = CWD

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)
print("Python:", sys.executable)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from IPython.display import display

from scripts import day2_quantification_helpers as dqh
from scripts import run_pixel_level_quantification as rpq

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

            ## Paths And Parameters

            This stage intentionally runs the current full quantification scaffold
            so later notebooks can stay focused on threshold review, containment, and density alternatives.
            

In [ ]:
DATA_DIR = ROOT / "data" / "with DAPI"
MANIFEST_OUTPUT = ROOT / "results" / "manifests" / "raw_input_manifest.tsv"
MASK_DIR = ROOT / "results" / "masks" / "dapi_whole_organoid_masks"
PLANE_METRICS_OUTPUT = ROOT / "results" / "tables" / "01_mask_and_plane_metrics.tsv"
THRESHOLD_OUTPUT = ROOT / "results" / "tables" / "02_reporter_thresholds.tsv"
AXIS_SCALE_OUTPUT = ROOT / "results" / "tables" / "02_reporter_axis_scales.tsv"
PLANE_SUMMARY_OUTPUT = ROOT / "results" / "tables" / "02_plane_reporter_summary.tsv"
STACK_SUMMARY_OUTPUT = ROOT / "results" / "tables" / "02_stack_reporter_summary.tsv"
PIXEL_SAMPLE_OUTPUT = ROOT / "results" / "tables" / "02_sampled_pixel_profiles.tsv"
CONTAINMENT_PLANE_OUTPUT = ROOT / "results" / "tables" / "03_plane_containment_by_method.tsv"
CONTAINMENT_STACK_OUTPUT = ROOT / "results" / "tables" / "03_stack_containment_by_method.tsv"
CONTAINMENT_METHOD_OUTPUT = ROOT / "results" / "tables" / "03_containment_method_summary.tsv"
DENSITY_OUTPUT = ROOT / "results" / "tables" / "04_density_maps.tsv"

SELECTED_THRESHOLD_METHOD_KEY = "snr_fixed_4sigma"
THRESHOLD_Z = 4.0
FIXED_SNR_THRESHOLD = 4.0
BACKGROUND_ESTIMATOR = "whole_off_organoid"
BACKGROUND_ANNULUS_INNER_PX = 6
BACKGROUND_ANNULUS_OUTER_PX = 20
BACKGROUND_MIN_RING_PIXELS = 1000
OUTPUT_SAMPLE_PER_STACK = 30000
DENSITY_BINS = 96
AXIS_SCALE_QUANTILE = 0.995

WRITE_OUTPUTS = True
RUN_PIPELINE = True

In [ ]:
if RUN_PIPELINE:
    quant_res = rpq.run_quantification_pipeline(
        root=ROOT,
        data_dir=DATA_DIR,
        manifest_output=MANIFEST_OUTPUT,
        mask_dir=MASK_DIR,
        plane_metrics_output=PLANE_METRICS_OUTPUT,
        threshold_output=THRESHOLD_OUTPUT,
        axis_scale_output=AXIS_SCALE_OUTPUT,
        plane_summary_output=PLANE_SUMMARY_OUTPUT,
        stack_summary_output=STACK_SUMMARY_OUTPUT,
        pixel_sample_output=PIXEL_SAMPLE_OUTPUT,
        containment_plane_output=CONTAINMENT_PLANE_OUTPUT,
        containment_stack_output=CONTAINMENT_STACK_OUTPUT,
        containment_method_output=CONTAINMENT_METHOD_OUTPUT,
        density_output=DENSITY_OUTPUT,
        selected_threshold_method_key=SELECTED_THRESHOLD_METHOD_KEY,
        threshold_z=THRESHOLD_Z,
        fixed_snr_threshold=FIXED_SNR_THRESHOLD,
        background_estimator=BACKGROUND_ESTIMATOR,
        background_annulus_inner_px=BACKGROUND_ANNULUS_INNER_PX,
        background_annulus_outer_px=BACKGROUND_ANNULUS_OUTER_PX,
        background_min_ring_pixels=BACKGROUND_MIN_RING_PIXELS,
        output_sample_per_stack=OUTPUT_SAMPLE_PER_STACK,
        density_bins=DENSITY_BINS,
        axis_scale_quantile=AXIS_SCALE_QUANTILE,
        write_outputs=WRITE_OUTPUTS,
    )
    manifest_df = quant_res["manifest_df"].copy().sort_values("file_id").reset_index(drop=True)
    plane_metrics_df = quant_res["plane_metrics_df"].copy().sort_values(["file_id", "z_index"]).reset_index(drop=True)
    print(quant_res["summary_text"])
else:
    manifest_df = pd.read_csv(MANIFEST_OUTPUT, sep="\t").sort_values("file_id").reset_index(drop=True)
    plane_metrics_df = pd.read_csv(PLANE_METRICS_OUTPUT, sep="\t").sort_values(["file_id", "z_index"]).reset_index(drop=True)

print("Manifest rows:", len(manifest_df))
print("Plane rows:", len(plane_metrics_df))
display(plane_metrics_df.head(20))

In [ ]:
mask_summary = (
    plane_metrics_df.groupby("file_id", as_index=False)
    .agg(
        position_label=("position_label", "first"),
        file_name=("file_name", "first"),
        z_planes=("z_index", "nunique"),
        mean_mask_fraction=("mask_fraction", "mean"),
        min_mask_fraction=("mask_fraction", "min"),
        max_mask_fraction=("mask_fraction", "max"),
        mean_mask_area_px=("mask_area_px", "mean"),
        mean_dapi_bg_sigma=("dapi_bg_sigma", "mean"),
    )
    .sort_values("file_id")
)
display(mask_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
for file_id, sub in plane_metrics_df.groupby("file_id", sort=True):
    label = sub["position_label"].iloc[0]
    axes[0].plot(sub["z_index"] + 1, sub["mask_fraction"], marker="o", linewidth=1.5, label=label)
    axes[1].plot(sub["z_index"] + 1, sub["mask_area_px"], marker="o", linewidth=1.5, label=label)

axes[0].set_title("Mask fraction across z")
axes[0].set_xlabel("z plane (1-based)")
axes[0].set_ylabel("mask fraction")
axes[1].set_title("Mask area across z")
axes[1].set_xlabel("z plane (1-based)")
axes[1].set_ylabel("mask area (px)")
axes[0].legend(ncol=2, fontsize=8, loc="upper right")
plt.show()

In [ ]:
def _robust_rescale(image: np.ndarray, q_low: float = 0.01, q_high: float = 0.995) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros_like(arr, dtype=np.float32)
    lo = float(np.quantile(finite, q_low))
    hi = float(np.quantile(finite, q_high))
    if not np.isfinite(lo):
        lo = 0.0
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0)


def plot_selected_review_plane_masks() -> None:
    if manifest_df.empty:
        print("Manifest is empty.")
        return

    fig, axes = plt.subplots(len(manifest_df), 5, figsize=(18, 3.5 * len(manifest_df)), constrained_layout=True)
    if len(manifest_df) == 1:
        axes = np.asarray([axes])

    for ax_row, row in zip(axes, manifest_df.itertuples(index=False)):
        stack = dqh.load_czi_stack(ROOT / row.file_path)
        z_index = int(row.selected_review_z_0based) if pd.notna(row.selected_review_z_0based) else int(stack.data_czyx.shape[1] // 2)
        mask_row = plane_metrics_df[
            (plane_metrics_df["file_id"] == int(row.file_id)) & (plane_metrics_df["z_index"] == z_index)
        ].iloc[0]
        mask = tifffile.imread(ROOT / mask_row["mask_path"]).astype(bool)

        bf = stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "brightfield"), z_index]
        dapi = stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "dapi"), z_index]
        foxf1 = stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "foxf1"), z_index]
        bmp4 = stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "bmp4"), z_index]

        composite = np.dstack([
            _robust_rescale(foxf1),
            _robust_rescale(bmp4),
            np.zeros_like(foxf1, dtype=np.float32),
        ])

        panels = [
            ("BF", _robust_rescale(bf), "gray"),
            ("DAPI", _robust_rescale(dapi), "gray"),
            ("DAPI + mask", _robust_rescale(dapi), "gray"),
            ("FOXF1/BMP4", composite, None),
            ("Mask only", mask.astype(float), "gray"),
        ]
        for ax, (title, img, cmap) in zip(ax_row, panels):
            if cmap is None:
                ax.imshow(img)
            else:
                ax.imshow(img, cmap=cmap)
            if title in {"DAPI + mask", "Mask only"}:
                ax.contour(mask.astype(float), levels=[0.5], colors="#00e5ff", linewidths=0.9)
            ax.set_title(
                f"{row.position_label} | z{z_index + 1} | {title}\n"
                f"frac={float(mask_row['mask_fraction']):.3f} | comps={int(mask_row['n_raw_components'])}"
            )
            ax.axis("off")

    plt.show()


plot_selected_review_plane_masks()

            ## All-z DAPI Mask Strips

            This section is intentionally verbose. It shows every z plane for every stack with the saved DAPI mask boundary overlaid,
            which is usually the fastest way to catch bad planes or over/undersegmentation.
            

In [ ]:
def plot_all_z_dapi_mask_strips() -> None:
    for row in manifest_df.itertuples(index=False):
        stack = dqh.load_czi_stack(ROOT / row.file_path)
        n_z = int(stack.data_czyx.shape[1])
        fig, axes = plt.subplots(1, n_z, figsize=(3.6 * n_z, 3.8), constrained_layout=True)
        if n_z == 1:
            axes = [axes]

        for z_index, ax in enumerate(axes):
            metric_row = plane_metrics_df[
                (plane_metrics_df["file_id"] == int(row.file_id)) & (plane_metrics_df["z_index"] == int(z_index))
            ].iloc[0]
            mask = tifffile.imread(ROOT / metric_row["mask_path"]).astype(bool)
            dapi = stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "dapi"), z_index]

            ax.imshow(_robust_rescale(dapi), cmap="gray")
            ax.contour(mask.astype(float), levels=[0.5], colors="#00e676", linewidths=1.0)
            review_tag = " | review" if bool(metric_row["is_selected_review_z"]) else ""
            ax.set_title(
                f"{row.position_label} | z{z_index + 1}{review_tag}\n"
                f"frac={float(metric_row['mask_fraction']):.3f}"
            )
            ax.axis("off")

        plt.show()


plot_all_z_dapi_mask_strips()